<a href="https://colab.research.google.com/github/korkutanapa/OEE_ARTICLE_STUDIES/blob/main/iam774hw2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

HOMEWORK II BY KORKUT ANAPA 774760

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Fix random seed for reproducibility
torch.manual_seed(42)

# 1. Define the Runge function with noise
def runge(x, tolerance=0.1):
    return 1 / (1 + 25 * x**2) + tolerance * torch.rand_like(x)

# 2. Sampling methods
def equidistant_sample(n):
    return torch.linspace(-1, 1, n)

def random_sample(n):
    return 2 * torch.rand(n) - 1  # Uniform from [-1, 1]

def chebyshev_sample(n):
    k = torch.arange(n)
    return torch.cos(k * np.pi / (n - 1))


In [ ]:
# 3. Define DNN model
class DNN(nn.Module):
    def __init__(self, layer_sizes):
        super(DNN, self).__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# 4. Training function
def train_model(x_train, y_train, x_val, y_val, hidden_layers, tolerance, epochs=500):
    model = DNN([1] + hidden_layers + [1])
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    # Reshape data
    x_train = x_train.unsqueeze(1)
    y_train = y_train.unsqueeze(1)
    x_val = x_val.unsqueeze(1)
    y_val = y_val.unsqueeze(1)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred = model(x_train)
        loss = loss_fn(y_pred, y_train)
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        y_val_pred = model(x_val)
        val_loss = loss_fn(y_val_pred, y_val).item()
    return val_loss, model

# 5. Experiment configuration
n_points = 100
tolerances = [1e-2, 1e-3, 1e-4]
hidden_configs = [[16, 16], [32, 32], [64, 64]]
sampling_methods = {
    "Equidistant": equidistant_sample,
    "Random": random_sample,
    "Chebyshev": chebyshev_sample
}

# 6. Run experiments and collect results
results = []

for tol in tolerances:
    for sampling_name, sampler in sampling_methods.items():
        x = sampler(n_points)
        y = runge(x, tolerance=tol)
        x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2)
        for config in hidden_configs:
            val_loss, _ = train_model(x_train, y_train, x_val, y_val, config, tolerance=tol)
            results.append({
                "Tolerance": tol,
                "Sampling": sampling_name,
                "Hidden Layers": config,
                "Validation MSE": val_loss
            })

# 7. (Optional) Print results
for r in results:
    print(r)